# pcm - Simulate continuous traits

Simulate a continuous trait evolved under one or more models of trait evolution (BM, OU, EB). 

For model fitting and AIC/AICc selection, see [fit continuous traits](pcm-fit-continuous.ipynb).


In [1]:
import toytree

In [2]:
# tree used in examples
tree = toytree.rtree.bdtree(ntips=24, seed=123)

## simulate_continuous_trait
The method ``simulate_continuous_bm`` can be called from the ``pcm`` module or tree-level APIs. Given the tree, a single continuous trait is simulated evolving along its edges. Three evolutionary models are supported: BM, OU, and EB (described below). Model parameters can be set uniformly across the tree, or applied piece-wise (Brownie-style) to different edges of the tree. The simulated trait is returned as a pandas.Series.

In [3]:
# simulate one trait using the module-level API
trait = toytree.pcm.simulate_continuous_trait(tree, params=1.0)

# simulate one trait using the tree-level API
trait = tree.pcm.simulate_continuous_trait(params=1.0)

In [4]:
# show the first few values
trait.head()

0   -1.748358
1    0.017737
2   -2.105518
3   -1.940644
4    0.263762
Name: X, dtype: float64

## Models
### BM
The amount of change in a continuous trait over a given time interval can be modeled under Brownian motion as the result of a random walk. At each time step the value changes by an amount randomly sampled from a normal distribution with mean=0 and variance described by an evolutionary rate parameter, $\sigma^2$. To model the change over an interval of time (e.g. a branch length) a random value is sampled from a normal distribution with mean=0 and variance as the product of the branch length and rate parameter ($\sigma^2 t$).

In [23]:
trait = tree.pcm.simulate_continuous_trait(params=1.0, model="BM")

### OU

In [12]:
trait = tree.pcm.simulate_continuous_trait(params=(1.0, 5.0), model="OU")

### EB

In [13]:
trait = tree.pcm.simulate_continuous_trait(params=(1.0, 0.5), model="EB")

## Parameters

### root_state

In [15]:
# set the name of the simulated trait
tree.pcm.simulate_continuous_trait(params=10.0, root_state=100).head()

0     98.952914
1    105.121784
2     97.271804
3     97.335345
4    107.083232
Name: X, dtype: float64

### name
The name of the returned Series can be set using the ``name`` argument. This is useful when simulating multiple traits and for downstream plotting.

In [55]:
# set the name of the simulated trait
tree.pcm.simulate_continuous_trait(params=1.0, name="body_size").head()

0    1.359598
1   -0.732873
2    1.135684
3    1.086460
4   -0.245482
Name: body_size, dtype: float64

### inplace
The returned data can be automatically assigned to nodes of the tree by using ``inplace=True``. This is convenient since assigning data to the tree is the recommended workflow for downstream visualization and analysis.

In [57]:
# store simulated trait to the tree as 'body_size' feature
tree.pcm.simulate_continuous_trait(params=1.0, name="body_size", inplace=True)

# extract data from the tree
tree.get_node_data("body_size").head()

0   -2.341439
1    0.967356
2    0.239255
3    0.751327
4    0.665639
dtype: float64

### tips_only

In [20]:
# only return simulated values for the tip nodes
trait = tree.pcm.simulate_continuous_trait(tips_only=True)
trait.size

24

### seed

### regime

In [24]:
toytree.pcm.simulate_continuous_trait??

Signature:
toytree.pcm.simulate_continuous_trait(
    tree: 'ToyTree',
    model: "Literal['bm', 'ou', 'eb']" = 'bm',
    params: 'float | tuple[float, float] | dict[str, float] | dict[str, tuple[float, float]]' = 1.0,
    root_state: 'float | None' = None,
    name: 'str' = 'X',
    tips_only: 'bool' = False,
    regime: 'str | pd.Series | None' = None,
    inplace: 'bool' = False,
    seed: 'int | np.random.Generator | None' = None,
) -> 'pd.Series'
Source:   
@add_subpackage_method(PhyloCompAPI)
# fmt: off
def simulate_continuous_trait(
    tree: ToyTree,
    model: Literal["bm", "ou", "eb"] = "bm",
    params: float | tuple[float, float] | dict[str, float] | dict[str, tuple[float, float]] = 1.0,  # noqa: E501
    root_state: float | None = None,
    name: str = "X",
    tips_only: bool = False,
    regime: str | pd.Series | None = None,
    inplace: bool = False,
    seed: int | np.random.Generator | None = None,
) -> pd.Series:
    # fmt: on
    """Simulate one continuous trait un

In [22]:
# # draw a tree showing trait values on edges
# canvas, axes, mark = tree.draw(layout='d', node_sizes=10 + (trait * 10), node_mask=False)
# tree.annotate.add_edges(
#     axes, width=5, use_color_gradient=True,
#     color=toytree.style.get_color_mapped_values(trait, "BlueRed"),
# );

In [ ]:
## Examples

In [7]:
# simulate three traits with different sigma2 params
toytree.pcm.simulate_continuous_bm(tree, sigma2=[1.0, 2.0, 3.0])

,t0,t1,t2
0,-1.472603,-0.800906,-0.530740
1,1.069024,-1.122443,1.965577
2,1.374335,-1.702547,1.019660
3,-0.707592,-0.427338,-0.981979
4,0.310147,1.468788,-0.471964
5,-0.307951,0.370279,-2.149104
6,0.898934,-0.694945,1.512550
7,-0.123471,-0.895061,0.735396
8,-0.557832,0.931584,-1.155385
9,-0.470610,0.167431,-0.866337


In [8]:
# use a dict to assign custom names to traits
toytree.pcm.simulate_continuous_bm(tree, sigma2={"size": 1.0, "speed": 5.0})

,size,speed
0,0.049582,1.817998
1,-1.423119,2.317576
2,-1.395612,3.161720
3,-1.030223,-2.718127
4,-0.252130,0.707599
5,-0.255603,0.721784
6,-0.882038,0.961516
7,-0.702029,-0.369989
8,-0.522694,1.239612
9,-0.859270,0.148724


#### tips_only
The data simulated above includes a trait value for every node in the tree, including internal nodes. However, in many cases we may be only interested in the traits at the tips of the tree. The argument `tips_only` will return on the simulated values for the tip nodes. (Note that the simulation process requires generating values for internal nodes, so you are effectively discarding that information when using this option, but it can be useful to keep things tidy). 

In [9]:
# simulate traits and store only for the tips
toytree.pcm.simulate_continuous_bm(tree, sigma2=[1.0], tips_only=True)

0   -0.933668
1   -0.235753
2   -0.790268
3    0.063142
4   -0.906436
5    0.619516
Name: t0, dtype: float64

#### root_states
You can set the root state for one or more simulated traits using the option `root_states`. The default root_state is 0. You can see this in the first few simulations above where the root node (node 10) has a value of 0.0 for each trait. Below we simulate the same tree traits but with different starting (root) values.

In [11]:
# simulate three traits with different sigma2 params
toytree.pcm.simulate_continuous_bm(
    tree, sigma2=[1.0, 2.0, 3.0], root_state=[10, 12, 50]
)

,t0,t1,t2
0,9.326036,12.925790,51.333027
1,9.900741,12.468017,49.848642
2,9.444084,11.583068,49.843916
3,8.769566,11.976691,50.319686
4,9.426614,14.082008,50.972181
5,8.849216,13.646420,49.526899
6,10.036243,12.901570,49.929488
7,10.046361,12.520192,51.788822
8,9.086064,12.921766,48.732241
9,9.346141,12.252227,49.483396


#### inplace
By default the simulate data are returned in a pandas DataFrame where the index corresponds to the numeric idx labels of Nodes in the tree. Alternatively, you can use `inplace=True` to store the simulated traits as one or more features saved to Nodes of the tree.

In [14]:
# save simulated traits to the ToyTree
tree.pcm.simulate_continuous_bm(sigma2=[1.0, 2.0], inplace=True)

# fetch simulated trait feature data from the tree
tree.get_node_data(["t0", "t1"])

,t0,t1
0,0.918660,-2.355651
1,1.318429,-1.340058
2,2.181314,-0.556105
3,-2.274030,0.507288
4,-0.554852,1.646986
5,0.995970,1.415032
6,0.741131,0.125105
7,0.897047,0.266611
8,-0.395872,1.728233
9,-0.181352,1.152845


One motivation for this option is that it makes it very easy to visualize the traits on a tree drawing, where you can select the traits by name rather than entering in the trait variable. Here we use color mapping to draw node colors scaled to the Greys colormap.

In [18]:
# draw the tree and show trait t0 values
tree.draw(
    node_sizes=10, node_colors=("t0", "Greys"), node_mask=False, label="trait 't0'"
);

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="300.0px" height="275.0px" viewBox="0 0 300.0 275.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t9b9ed93de6894724b717ad3a3e373085"> r0 r1 r2 r3 r4 r5 trait 't0'

## Multivariate Brownian motion
To simulate traits with correlated evolution you can enter a variance-covariance matrix for the `rates` option. This can be be a list of lists, numpy array, or pandas DataFrame. 

In [11]:
# TODO